In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk


In [49]:
df = pd.read_csv("Kenya_Coffee_Export_Exchange_Rate.csv")

#checking if df has been loaded
# print(df)

#looking at the heads we are working with
columns = df.columns
print(columns)

#checking how the data looks like in the tables
head = df.head(10)
print(head)

# checking the number non-null values in the columns
df.describe()

# the shape of the data
shape = df.shape
print(shape)

# checking the data types of the columns
dtypes = df.dtypes
print(dtypes)

Index(['year', 'average_kenyan_coffee_price_usd',
       'average_foreign_coffee_price_usd', 'current_total_factor_productivity',
       'germany_import_coffee_price_usd', 'belgium_import_coffee_price_usd',
       'united_states_import_coffee_price_usd',
       'south_korea_import_coffee_price_usd', 'annual_coffee_production',
       'annual_exportable_coffee', 'annual_coffee_exports',
       'rashid_moledina_exports', 'kenya_cooperative_coffee_exporters_exports',
       'dormans_exports', 'mwangi_coffee_exporters_exports',
       'sannex_coffee_exports', 'africoff_trading_exports',
       'diamond_coffee_company_exports', 'rockbern_coffee_group_exports',
       'real_exchange_rate_percent', 'nominal_exchange_rate_percent',
       'average_agricultural_price', 'total_annual_imports',
       'total_annual_exports', 'real_gdp_growth_percent',
       'gross_capital_formation_percent', 'population_growth_rate_percent',
       'real_interest_rate_percent', 'natural_resource_endowment'],
   

In [50]:
# checking for null values in the columns
print(df.isnull().sum())

# drop all unecessary columns that are not needed for the analysis
df = df.drop(columns=[
    "germany_import_coffee_price_usd", "belgium_import_coffee_price_usd", "united_states_import_coffee_price_usd",
    "south_korea_import_coffee_price_usd", "rashid_moledina_exports", "kenya_cooperative_coffee_exporters_exports",
    "dormans_exports", "mwangi_coffee_exporters_exports", "sannex_coffee_exports",
    "africoff_trading_exports", "diamond_coffee_company_exports", "rockbern_coffee_group_exports","nominal_exchange_rate_percent","annual_exportable_coffee"
])

print(df.columns)
print(df.shape)
#  the 8 individual columns were dropped as they are not real independent data every single 
# one of them has a fixed propotion of the tottal exports and applied indetically across all the years. 
# The columns were dropped to avoid multicollinearity in the model and to avoid overfitting the model.

year                                          0
average_kenyan_coffee_price_usd               0
average_foreign_coffee_price_usd              0
current_total_factor_productivity             0
germany_import_coffee_price_usd               0
belgium_import_coffee_price_usd               0
united_states_import_coffee_price_usd         0
south_korea_import_coffee_price_usd           0
annual_coffee_production                      0
annual_exportable_coffee                      0
annual_coffee_exports                         0
rashid_moledina_exports                       0
kenya_cooperative_coffee_exporters_exports    0
dormans_exports                               0
mwangi_coffee_exporters_exports               0
sannex_coffee_exports                         0
africoff_trading_exports                      0
diamond_coffee_company_exports                0
rockbern_coffee_group_exports                 0
real_exchange_rate_percent                    0
nominal_exchange_rate_percent           

## observation on data cleaning


## FEATURE ENGINEERING
New columns that will fit the model properly before modelling


In [54]:


# step 1 : price gap in USD how competitely priced is kenya coffee vs the international market.
# Positive gap → foreign buyers pay more elsewhere → Kenya is cheaper → demand advantage
# Negative gap → Kenya prices its coffee above foreign market → possible demand loss

df['price_gap_usd'] = (
    df['average_foreign_coffee_price_usd'] - df['average_kenyan_coffee_price_usd']
).round(3)


# in 9 of 20 years kenyay's coffe was priced above the international market price. 
# In 11 of the 20 years kenya's coffee was priced below the international market price. 
# The average price gap is -0.13 USD which means that on average kenya's coffee is priced below the international market price by 0.13 USD.
# meaning kenya coffee commanded a quality premium as the gap turnd positive from 2005 onwards.
# reflecting  the rise in commodity price ad growing coffee demand in the international market.


#STEP 2: THE VOLUME INTERACTIONS
df['price_volume_interaction'] = (df['average_kenyan_coffee_price_usd'] * df['annual_coffee_exports']).round(3)

# approimately a revenue of 1.5 billion USD was generated from coffee exports in 2022, which is a 3.5% increase from the previous year.
# with a range of KES 1.2 billion to KES 1.5 billion in the last 20 years, the average revenue generated from coffee exports is KES 1.3 billion.
# the 2010 and 2011 comodity rice spie drove revenue from 37k to 114k in 2 years the crashed back.

# STEP 3: EXPORT INTENSITY WHAT SHARE OF KENYA'S COFFEE PRODUCTION GETS EXPORTED
#values > 1 export more than produced, values < 1 export less than produced, values = 1 export exactly what is produced
df['export_intensity'] = (df['annual_coffee_exports'] / df['annual_coffee_production']).round(3)

# this flags anamolous years where kenya exported more coffee than it produced, which is not possible.

#STEP 4: TREND INDEX THE GROWTH OF KENYA'S COFFEE EXPORTS OVER TIME
df['trend_index'] = df['year'] - df['year'].min() + 1 
# growthof the exports  for 20 years starting from 1 for the first year in the dataset  
#trend_index is useful: When detrending your data first 
# when you want to explicitly model "how much does each additional year of sector development add to exports"

# STEP 5: year on year growth of coffee exports
# how fast is kenya's coffee exports growing year on year, this is a measure of the growth of kenya's coffee exports over time.
df['yoy_growth'] = df['average_kenyan_coffee_price_usd'].pct_change() * 100
df['exchange_rate_yoy_pct'] = df['real_exchange_rate_percent'].pct_change() * 100
df['foreign_price_yoy_pct'] = df['average_foreign_coffee_price_usd'].pct_change() * 100

# the weak correlations with the target (all under ±0.14) suggest that the rate of price change matters less than the level for predicting total exports. 
# Lasso will likely zero these out — but including them lets the model confirm this empirically rather than us assuming it.


# STEP 6: LAST YEARS EXPORT VALUE AND PRICE
df['lag1_total_exports'] = df['total_annual_exports'].shift(1)
df['lag1_kenya_price'] = df['average_kenyan_coffee_price_usd'].shift(1)

# lag 1 total exports is the stronest predictor of total exports, and lag 1 kenya price is the second strongest predictor of total exports.
# it captures exports momentum years that follow with strong momentum tend to to be strong themselves


# STEP 7: PRODUCTION GAP 
df['production_gap'] = (df['annual_coffee_production'] - df['annual_coffee_exports']).round(3)

# 2003 had the worst stockpile drawdown 0f -247 kenya exported 247 more bags than it produced
# drawing heavily on reserves likely because global prices rose that year making it  profitable to export more than was produced.
# helps the model to distinguish between years hen exports are high because of strong reflecting geniuneproduction vs  stockpile liquidation.

# STEP 8: TRADE OPENNESS
df['trade_openness'] = df['total_annual_imports'] + df['total_annual_exports']
# used for statistical control of the overall size of the economy, which can affect exports through many channels.
# CANNOT BE USED AS A PREDICTOR OF EXPORTS, AS IT IS A FUNCTION OF EXPORTS.

# STEP 9: CREDIT CONDITION
df['credit_condition'] = df['real_interest_rate_percent']

# high credit conditions (low interest rates) make it cheaper to borrow money to finance exports, which can increase exports.
# real interest rates are negatively correlated with exports, as expected. The correlation is weak, but it is still useful to include in the model as a control variable.
# negating the rate we get a credit condition index that is higher better for exports maing the model coefficient positive and economcally interpretable.



print(df.head(10))
df = df.dropna().reset_index(drop=True)
print(df.head(10))
print(df.shape)



   year  average_kenyan_coffee_price_usd  average_foreign_coffee_price_usd  \
0  2001                        69.400000                             37.05   
1  2002                        67.670000                             30.91   
2  2003                        41.070000                             42.82   
3  2004                        71.010000                             56.33   
4  2005                        70.079945                             87.09   
5  2006                        76.904891                             87.02   
6  2007                        72.209092                             98.30   
7  2008                        71.608141                            109.26   
8  2009                        70.624260                            100.80   
9  2010                       139.971150                            134.00   

   current_total_factor_productivity  annual_coffee_production  \
0                           0.516281                       991   
1        

##  DATA MODELLING
since the dataset is a small datset this reinforces the use of LOOCV rather than train/split  it uses all the 19 rows for both training and validation.
nd use the regularised regression lasso